# Step 02. Dijkstra 알고리즘

> **출처**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)  
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

연속 공간을 격자로 이산화한 뒤, **비용이 가장 낮은 노드부터 확장**하는 Dijkstra로 최단 경로를 찾습니다.

> 이번 노트북은 **큰 함수를 한 번에 보지 않고**, 작은 조각(자료구조 → is_valid → 초기화 → *한 번의 확장* → 역추적)을 하나씩 이해한 뒤 마지막에 조립합니다.

## 학습 목표

1. Grid Map / 장애물 팽창 이해
2. Open Set(프론티어) / Closed Set(방문 완료) 개념
3. **'한 번의 확장'** 을 직접 보고, 전체 탐색이 그 반복임을 이해
4. 경로 역추적 + 탐색 과정 애니메이션

In [ ]:
%matplotlib inline
import sys, os, math, heapq
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

from utils.grid_map import make_simple_map, build_obstacle_map
from utils.visualization import plot_grid_map
from utils.grid_search import (calc_grid_index, calc_grid_position,
                               node_key, get_motion_model, is_valid, reconstruct_path)
from utils.search_viz import animate_motion_model, animate_search

print('라이브러리 로드 완료')


## 1. Grid Map 생성

### 이론 배경

Dijkstra는 **그래프** 위에서 동작합니다. 연속 공간을 격자로 바꿔 각 셀을 노드로 삼습니다.  
출발(-5,-5)에서 목표(50,50)까지 벽을 피해 경로를 찾습니다.

In [ ]:
ox, oy, sx, sy, gx, gy = make_simple_map()
print(f'장애물 {len(ox)}개,  출발 ({sx},{sy})  ->  목표 ({gx},{gy})')

fig, ax = plt.subplots(figsize=(6, 6))
plot_grid_map(ox, oy, sx, sy, gx, gy, ax=ax)
ax.set_title('Simple Grid Map')
plt.tight_layout(); plt.show()


## 2. 격자 파라미터

### 이론 배경

**Resolution**: 격자 한 칸 크기(작을수록 정밀·느림).  
**Robot Radius**: 장애물을 이만큼 팽창시켜 차량을 점으로 단순화 (C-Space 변환).

아래에서 **해상도 1m / 2m / 4m** 격자를 나란히 그려 셀 크기·개수 변화를 직접 비교합니다.

In [ ]:
RESOLUTION   = 2.0  # 격자 한 칸 크기 [m]
ROBOT_RADIUS = 1.0  # 장애물 팽창 반경 [m]

min_x, max_x = round(min(ox)), round(max(ox))
min_y, max_y = round(min(oy)), round(max(oy))
x_width = round((max_x - min_x) / RESOLUTION)
y_width = round((max_y - min_y) / RESOLUTION)

# 해상도가 격자 크기/개수에 미치는 영향을 눈으로 비교
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
for ax, res in zip(axes, [1.0, 2.0, 4.0]):
    xw = round((max_x - min_x) / res); yw = round((max_y - min_y) / res)
    for x in np.arange(min_x, max_x + res, res): ax.axvline(x, color='lightgray', lw=0.5)
    for y in np.arange(min_y, max_y + res, res): ax.axhline(y, color='lightgray', lw=0.5)
    ax.plot(ox, oy, '.k', ms=1.5, alpha=0.5)
    ax.plot(sx, sy, 'og', ms=9); ax.plot(gx, gy, 'xb', ms=9, mew=2)
    used = '   <-- USED' if res == RESOLUTION else ''
    ax.set_aspect('equal'); ax.grid(False)
    ax.set_title(f'resolution = {res} m{used}\n{xw} x {yw} = {xw*yw:,} cells', fontsize=10)
plt.suptitle('Resolution vs Grid Granularity  (finer = more cells = precise but slow)',
             fontsize=12)
plt.tight_layout(); plt.show()

print(f'선택: 해상도 {RESOLUTION} m -> {x_width} x {y_width} = {x_width*y_width:,} cells')
print('  해상도 1m: 4배 촘촘(정밀,느림) / 4m: 1/4로 성김(거침,빠름)')


## 3. 장애물 맵 (팽창)

### 이론 배경

`build_obstacle_map`이 각 격자 셀을 **통행 가능(free)** 또는 **장애물(blocked)** 로 분류합니다.  
robot_radius가 클수록 안전하지만 좁은 통로가 막힐 수 있습니다.

In [ ]:
# 장애물을 robot_radius 만큼 팽창시켜 통행 가능/불가 격자를 만든다
obstacle_map, min_x, min_y, max_x, max_y, x_width, y_width = \
    build_obstacle_map(ox, oy, RESOLUTION, ROBOT_RADIUS)

free_x, free_y, blk_x, blk_y = [], [], [], []
for ix in range(x_width):
    for iy in range(y_width):
        px, py = ix*RESOLUTION+min_x, iy*RESOLUTION+min_y
        if obstacle_map[ix][iy]: blk_x.append(px); blk_y.append(py)
        else:                    free_x.append(px); free_y.append(py)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(free_x, free_y, 's', color='lightblue', ms=6, alpha=0.6, label=f'Free ({len(free_x)})')
ax.plot(blk_x, blk_y, 's', color='dimgray', ms=6, alpha=0.8, label=f'Blocked ({len(blk_x)})')
ax.plot(sx, sy, 'og', ms=12); ax.plot(gx, gy, 'xb', ms=12, mew=2.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.2); ax.legend(loc='upper left', fontsize=8)
ax.set_title(f'Inflated Obstacle Map (robot_radius={ROBOT_RADIUS}m)')
plt.tight_layout(); plt.show()
print(f'통행 가능 {len(free_x)}개 / 장애물(팽창) {len(blk_x)}개')


## 4. Node — 상태 표현

### 이론 배경

노드 하나 = 탐색 상태 하나. 저장 정보:

| 필드 | 의미 |
|------|------|
| `x, y` | 격자 인덱스 |
| `cost` | 출발부터 누적 비용 g(n) |
| `parent_index` | 직전 노드 키 (역추적용) |

In [ ]:
class Node:
    # 격자 탐색의 기본 단위. (x, y)는 실제 좌표가 아닌 격자 인덱스.
    def __init__(self, x, y, cost, parent_index):
        self.x = x                     # 격자 인덱스
        self.y = y
        self.cost = cost               # 출발점부터 누적 비용 g(n)
        self.parent_index = parent_index  # 경로 역추적용 부모 키
    def __lt__(self, other):
        return self.cost < other.cost  # heapq 비교용

n = Node(5, 3, 7.07, parent_index=10)
print(f'예시: Node(x={n.x}, y={n.y}, cost={n.cost}, parent={n.parent_index})')


## 5. 좌표 변환 헬퍼

### 이론 배경

실제 좌표 ↔ 격자 인덱스 ↔ 1D 키 변환은 반복되는 기계적 계산이라 `utils/grid_search.py`로 분리했습니다. 노트북은 핵심 로직에 집중합니다.

```
calc_grid_index : 좌표[m] -> 인덱스
calc_grid_position : 인덱스 -> 좌표[m]
node_key : (ix,iy) -> 1D 정수 키
```

In [ ]:
# utils.grid_search 의 좌표 변환 함수 3종 (반복되는 기계적 계산이라 분리)
print(f'{"함수":<34}{"결과":>8}')
print('-'*44)
print(f'{"calc_grid_index(0.0, min_x, res)":<34}{calc_grid_index(0.0, min_x, RESOLUTION):>8}')
print(f'{"calc_grid_position(5, min_x, res)":<34}{calc_grid_position(5, min_x, RESOLUTION):>8.1f}')
print(f'{"node_key(Node(5,3,..))":<34}{node_key(n, x_width, min_x, min_y):>8}')
print()
print('node_key: 2D (ix,iy) -> 1D 정수 (dict 키로 사용)')


## 6. 이동 모델 (Motion Model)

### 이론 배경

그래프의 **엣지**. 8방향 이동, 비용은 유클리드 거리 (직선 1.0, 대각선 √2).  
√2를 써야 등방성(거리 왜곡 없음)이 보장됩니다.

In [ ]:
motion = get_motion_model()
DIRS = ['R','L','U','D','DL','UL','DR','UR']
print('8방향 이동 모델 (직선 1.0, 대각선 √2)')
for label, (dx, dy, cost) in zip(DIRS, motion):
    kind = 'straight' if cost == 1.0 else 'diagonal'
    print(f'  {label:<3} dx={dx:>2} dy={dy:>2} cost={cost:.3f} ({kind})')


In [ ]:
# 8방향을 하나씩 화살표로 (utils.search_viz)
HTML(animate_motion_model(motion, DIRS).to_jshtml())


## 7. is_valid — 통행 가능한 이웃인가?

### 이론 배경

이웃 노드를 만들 때마다 **격자 범위 안 + 장애물 아님**을 확인해야 합니다.  
`is_valid`가 이 두 조건을 검사합니다. 출발 노드의 8이웃에 적용해 봅니다.

In [ ]:
# 한 노드의 8방향 이웃이 통행 가능한지 검사 (범위 밖 / 장애물이면 불가)
start = Node(calc_grid_index(sx, min_x, RESOLUTION),
             calc_grid_index(sy, min_y, RESOLUTION), 0.0, -1)

print('출발 노드의 8이웃 검사:')
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(ox, oy, '.k', ms=2, alpha=0.4)
ax.plot(sx, sy, 'og', ms=12, zorder=5)
for (dx, dy, cost), label in zip(motion, DIRS):
    nb = Node(start.x+dx, start.y+dy, cost, -1)
    ok = is_valid(nb, obstacle_map, min_x, min_y, max_x, max_y, RESOLUTION)
    px = calc_grid_position(nb.x, min_x, RESOLUTION)
    py = calc_grid_position(nb.y, min_y, RESOLUTION)
    ax.plot(px, py, 'o', color='limegreen' if ok else 'red', ms=10, zorder=4)
    print(f'  {label:<3} -> {"통행 가능" if ok else "불가(범위밖/장애물)"}')
ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
ax.set_title('Neighbors of Start (green=valid, red=blocked)')
plt.tight_layout(); plt.show()


## 8. Open / Closed Set 초기화

### 이론 배경

- **Open Set**: 탐색 예정 노드(프론티어). 처음엔 출발 노드 1개.
- **Closed Set**: 비용이 확정된 노드(다시 안 봄).
- **Priority Queue**: 비용 최소 노드를 빠르게 꺼내는 min-heap.

In [ ]:
# Open Set  : 탐색 예정(프론티어),  Closed Set: 비용 확정(방문 완료)
goal = Node(calc_grid_index(gx, min_x, RESOLUTION),
            calc_grid_index(gy, min_y, RESOLUTION), 0.0, -1)

open_set   = {node_key(start, x_width, min_x, min_y): start}   # 출발 노드만 넣고 시작
closed_set = {}
priority_queue = [(0.0, start)]    # (cost, node)  min-heap

print(f'초기 상태:  Open Set {len(open_set)}개(출발),  Closed Set {len(closed_set)}개')
print('이제 Open Set에서 비용이 가장 낮은 노드를 꺼내 확장하는 것을 반복한다')


## 9. ★ 확장을 여러 단계 진행 (핵심)

### 이론 배경

Dijkstra의 한 스텝:

```
① Open Set에서 비용 최소 노드 current 를 꺼낸다
② current 를 Closed Set으로 옮긴다
③ current 의 8이웃 중 (통행 가능 + 미방문 + 더 싼 경로)만 Open Set에 추가
```

이 한 스텝을 **여러 번 반복**하며 프론티어(Open Set)가 어떻게 퍼지는지 단계별로 봅니다.  
- 회색 = Closed Set(방문 완료) / 하늘색 = Open Set(다음에 볼 후보) / ★ = 이번에 꺼낸 노드
- step이 진행될수록 회색이 늘고 프론티어가 **방사형으로 확장**됩니다.

전체 탐색은 이 과정을 목표에 도달할 때까지 반복할 뿐입니다.

In [ ]:
# 섹션 8 자료구조를 처음 상태로 되돌리고 '한 번의 확장'을 여러 번 반복
open_set   = {node_key(start, x_width, min_x, min_y): start}
closed_set = {}
priority_queue = [(0.0, start)]

def expand_once():
    # Open Set에서 비용 최소 노드를 꺼내 8이웃을 Open에 추가하고 그 노드를 반환
    while priority_queue:
        _, cur = heapq.heappop(priority_queue)
        cid = node_key(cur, x_width, min_x, min_y)
        if cid not in open_set: continue
        open_set.pop(cid); closed_set[cid] = cur
        for dx, dy, cost in motion:
            nb = Node(cur.x+dx, cur.y+dy, cur.cost+cost, cid)
            nk = node_key(nb, x_width, min_x, min_y)
            if not is_valid(nb, obstacle_map, min_x, min_y, max_x, max_y, RESOLUTION): continue
            if nk in closed_set or (nk in open_set and open_set[nk].cost <= nb.cost): continue
            open_set[nk] = nb; heapq.heappush(priority_queue, (nb.cost, nb))
        return cur

gx_ = lambda n: calc_grid_position(n.x, min_x, RESOLUTION)
gy_ = lambda n: calc_grid_position(n.y, min_y, RESOLUTION)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for step, ax in enumerate(axes.flat, 1):
    cur = expand_once()
    ax.plot(ox, oy, '.k', ms=1, alpha=0.3)
    ax.plot([gx_(n) for n in open_set.values()],   [gy_(n) for n in open_set.values()],   's', color='deepskyblue', ms=8, alpha=0.7)
    ax.plot([gx_(n) for n in closed_set.values()], [gy_(n) for n in closed_set.values()], 's', color='gray', ms=8, alpha=0.85)
    ax.plot(gx_(cur), gy_(cur), '*', color='gold', ms=20, zorder=6)
    ax.plot(sx, sy, 'og', ms=8); ax.plot(gx, gy, 'xb', ms=8, mew=2)
    ax.set_aspect('equal'); ax.set_xlim(-12, 22); ax.set_ylim(-12, 22)
    ax.set_title(f'step {step}:  closed={len(closed_set)}, open={len(open_set)}', fontsize=10)
for col, lab in [('gray','Closed (visited)'), ('deepskyblue','Open (frontier)'), ('gold','current (*)')]:
    axes[0,0].plot([], [], 's' if col!='gold' else '*', color=col, label=lab)
axes[0,0].legend(loc='upper left', fontsize=8)
plt.suptitle('Frontier grows step by step (first 6 expansions, zoomed near Start)', fontsize=13)
plt.tight_layout(); plt.show()
print('단계마다 current(*)를 Closed로 옮기고 이웃을 Open에 추가 -> 프론티어가 방사형으로 확장')


## 10. 전체 루프 조립

### 이론 배경

지금까지의 조각(초기화 → 한 번의 확장 → 목표 확인 → 역추적)을 `while` 루프로 묶습니다.  
**최적성**: 비용이 양수라 Closed Set에 들어간 노드의 비용은 더 줄지 않습니다.  
**복잡도**: O((V+E) log V).

In [ ]:
def dijkstra(sx, sy, gx, gy, ox, oy, resolution, robot_radius):
    obstacle_map, min_x, min_y, max_x, max_y, x_width, y_width = \
        build_obstacle_map(ox, oy, resolution, robot_radius)

    start = Node(calc_grid_index(sx, min_x, resolution),
                 calc_grid_index(sy, min_y, resolution), 0.0, -1)
    goal  = Node(calc_grid_index(gx, min_x, resolution),
                 calc_grid_index(gy, min_y, resolution), 0.0, -1)

    open_set   = {node_key(start, x_width, min_x, min_y): start}
    closed_set = {}
    pq = [(0.0, start)]
    explore_order = []   # 시각화용 탐색 순서

    while pq:
        _, current = heapq.heappop(pq)
        cid = node_key(current, x_width, min_x, min_y)
        if cid not in open_set: continue          # 이미 처리된 중복 항목
        open_set.pop(cid); closed_set[cid] = current
        explore_order.append((calc_grid_position(current.x, min_x, resolution),
                              calc_grid_position(current.y, min_y, resolution), current.cost))
        if current.x == goal.x and current.y == goal.y:   # 목표 도달
            goal.parent_index = current.parent_index; goal.cost = current.cost
            break
        for dx, dy, cost in get_motion_model():           # 8이웃 확장 (위 '한 번의 확장')
            nb = Node(current.x+dx, current.y+dy, current.cost+cost, cid)
            nk = node_key(nb, x_width, min_x, min_y)
            if not is_valid(nb, obstacle_map, min_x, min_y, max_x, max_y, resolution): continue
            if nk in closed_set: continue
            if nk not in open_set or open_set[nk].cost > nb.cost:
                open_set[nk] = nb; heapq.heappush(pq, (nb.cost, nb))

    path_x, path_y = reconstruct_path(goal, closed_set, min_x, min_y, resolution)   # 역추적
    closed_x = [calc_grid_position(n.x, min_x, resolution) for n in closed_set.values()]
    closed_y = [calc_grid_position(n.y, min_y, resolution) for n in closed_set.values()]
    return path_x, path_y, closed_x, closed_y, explore_order

print('dijkstra() 정의 완료 — 위에서 본 조각들(초기화/한번의 확장/역추적)의 조립')


## 11. 탐색 실행

### 이론 배경

탐색 노드 수(전체 셀 대비 %)가 효율 지표입니다. 경로 길이는 A*와 비교해도 동일해야 합니다.

In [ ]:
import time
t0 = time.time()
path_x, path_y, closed_x, closed_y, explore_order = \
    dijkstra(sx, sy, gx, gy, ox, oy, RESOLUTION, ROBOT_RADIUS)
dt = (time.time()-t0)*1000

path_len = sum(math.hypot(path_x[i+1]-path_x[i], path_y[i+1]-path_y[i]) for i in range(len(path_x)-1))
print(f'탐색 노드 {len(closed_x):,}개  ({len(closed_x)/x_width/y_width*100:.0f}% of {x_width*y_width:,})')
print(f'경로 길이 {path_len:.1f} m,  탐색 시간 {dt:.0f} ms')


## 12. 탐색 과정 애니메이션

### 이론 배경

색(plasma)이 낮은 비용(보라)부터 퍼집니다. 목표 방향을 모르므로 **방사형**으로 확산 → Dijkstra의 비효율 원인.  
Step 03 A*에서 이 모양이 어떻게 바뀌는지 비교하게 됩니다.

In [ ]:
# 비용(plasma) 순으로 퍼지다 목표 도달 후 경로를 역추적 (utils.search_viz)
HTML(animate_search(explore_order, path_x, path_y, ox, oy, sx, sy, gx, gy,
                    title='Dijkstra').to_jshtml())


## 13. 결과 요약

### Dijkstra 특징

| 항목 | 내용 |
|------|------|
| 최적성 | 보장 (비용 ≥ 0) |
| 완전성 | 보장 |
| 방향성 | 없음 → 전방위 방사형 |
| 한계 | 목표와 무관한 노드도 탐색 |

→ **Step 03 A***: 우선순위를 `g(n)` → `g(n)+h(n)`로 바꿔 목표 방향 집중.

In [ ]:
from utils.visualization import plot_search_result
fig = plot_search_result(closed_x, closed_y, path_x, path_y,
                         ox, oy, sx, sy, gx, gy, title=f'Dijkstra ({len(closed_x):,} nodes)')
plt.tight_layout(); plt.show()
print('Dijkstra: 방사형 전방위 탐색 -> 최적 경로 보장, 단 목표와 무관한 노드도 많이 탐색')
print('다음 Step 03 A*: Heuristic을 더해 목표 방향으로 집중 (탐색 노드 감소)')
